# 🔎 Cortador Auditável em Tela Dividida (`#DesafioProvaReal`)
**Sem instalar nada no seu computador!** Preencha os campos abaixo com os dados validados pelo **Prompt 01** e clique no botão **Play (▶)** ao lado da célula para gerar seu vídeo vertical `9:16` (`1080x1920`) com:
- Velocidade da fala travada em `1.0x` (conformidade Resolução TSE nº 23.610/2019 — zero *cheapfake*);
- Painel Editorial Superior com data original da gravação + veredito e fonte oficial;
- Vídeo original íntegro na metade inferior + Ficha de Auditoria pronta para colar no primeiro comentário!

In [ ]:
# @title 🎬 Configurar e Gerar Corte Verificado
URL_FONTE_ORIGINAL = "https://www.youtube.com/watch?v=EXEMPLO" # @param {type:"string"}
INICIO_SEGUNDOS = 60.0 # @param {type:"number"}
FIM_SEGUNDOS = 105.0 # @param {type:"number"}
DATA_ORIGINAL_GRAVACAO = "24/10/2026" # @param {type:"string"}
SELO_VEREDITO = "FATO COMPROVADO" # @param ["FATO COMPROVADO", "INCONCLUSIVO", "FORA DE CONTEXTO"]
RESUMO_FONTE_OFICIAL = "Dados Abertos Câmara dos Deputados / IBGE" # @param {type:"string"}
MODELO_IA_USADO = "ChatGPT (GPT-4o)" # @param {type:"string"}
CUSTO_ESTIMADO_REAIS = "0,08" # @param {type:"string"}
PROMPT_DECLARADO = "Extrair falas objetivas sobre economia confrontando com dados oficiais." # @param {type:"string"}

import subprocess, hashlib, shlex
from pathlib import Path

assert FIM_SEGUNDOS > INICIO_SEGUNDOS, 'O tempo final deve ser maior que o inicial.'
assert (FIM_SEGUNDOS - INICIO_SEGUNDOS) <= 180, 'Corte máximo recomendado: 180 segundos.'

subprocess.run(['pip', 'install', '-q', 'yt-dlp', 'pillow'], check=True)

print('1/3 Baixando trecho original sem alterar velocidade (1.0x)...')
subprocess.run([
    'yt-dlp', '-f', 'mp4/best',
    '--download-sections', f'*{INICIO_SEGUNDOS}-{FIM_SEGUNDOS}',
    '-o', 'trecho_bruto.mp4', URL_FONTE_ORIGINAL
], check=True)

from PIL import Image, ImageDraw, ImageFont
cores = {
    'FATO COMPROVADO': '#00E5FF',
    'INCONCLUSIVO': '#FFB300',
    'FORA DE CONTEXTO': '#FF5252'
}
cor = cores.get(SELO_VEREDITO, '#00E5FF')
painel = Image.new('RGB', (1080, 640), '#0D1117')
dr = ImageDraw.Draw(painel)
dr.rectangle([40, 40, 1040, 600], outline=cor, width=4)
dr.text((80, 90), f'PROVA REAL • {SELO_VEREDITO}', fill=cor)
dr.text((80, 180), f'Data Real da Fala: {DATA_ORIGINAL_GRAVACAO}', fill='#F0F6FC')
dr.text((80, 260), f'Fonte Oficial: {RESUMO_FONTE_OFICIAL[:58]}', fill='#F0F6FC')
dr.text((80, 520), 'Audite no repositório aberto #DesafioProvaReal (Velocidade 1.0x)', fill='#8B949E')
painel.save('painel_topo.png')

print('2/3 Renderizando Tela Dividida Editorial 1080x1920...')
subprocess.run([
    'ffmpeg', '-y', '-loop', '1', '-i', 'painel_topo.png', '-i', 'trecho_bruto.mp4',
    '-filter_complex',
    '[1:v]scale=1080:1280:force_original_aspect_ratio=decrease,pad=1080:1280:(ow-iw)/2:(oh-ih)/2:#0D1117[vid];[0:v][vid]vstack=inputs=2[outv]',
    '-map', '[outv]', '-map', '1:a?', '-shortest', '-map_metadata', '-1',
    '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-c:a', 'aac', 'corte_prova_real.mp4'
], check=True)

ficha = f"""🔎 FICHA DE AUDITORIA — #DesafioProvaReal
📌 Fonte Original: {URL_FONTE_ORIGINAL}
⏱️ Trecho Exato Extraído: {INICIO_SEGUNDOS:.2f}s até {FIM_SEGUNDOS:.2f}s | Data Original: {DATA_ORIGINAL_GRAVACAO}
📊 Veredito das Fontes: {SELO_VEREDITO} — {RESUMO_FONTE_OFICIAL}
🤖 Modelo de IA Utilizado: {MODELO_IA_USADO}
🧮 Custo aprox.: R$ {CUSTO_ESTIMADO_REAIS}
📝 Prompt Integral Usado pelo Autor:
"{PROMPT_DECLARADO}"

⚠️ AVISO ANTI-RASURA: Se este comentário aparecer marcado como 'editado', esta postagem perde automaticamente a validade e o autor é desclassificado do ranking. Caso esta rede social não exiba histórico público de edição nos comentários, NÃO CONFIE CEGAMENTE: copie este comentário e o link da fonte original e valide agora no Prompt Auditor gratuito no repositório oficial do #DesafioProvaReal!"""

Path('comentario_oficial.txt').write_text(ficha, encoding='utf-8')
print('\n3/3 CONCLUÍDO! Copie o Comentário Oficial abaixo sem editar nada:\n')
print(ficha)
try:
    from google.colab import files
    files.download('corte_prova_real.mp4')
except Exception:
    print('Arquivo salvo localmente como corte_prova_real.mp4')